In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add project root to Python path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

In [2]:
from src.io import load_yaml, load_pcic_netcdf, load_station_csv, resolve_relative_path

paths = load_yaml("config/paths.yml")

### Load the NetCDF file from PCIC

In [3]:
ds = load_pcic_netcdf(paths["raw_data"]["pcic_precip"])
ds

<xarray.Dataset> Size: 50GB
Dimensions:  (lat: 510, lon: 1068, time: 23011)
Coordinates:
  * lat      (lat) float64 4kB 41.04 41.12 41.21 41.29 ... 83.29 83.38 83.46
  * lon      (lon) float64 9kB -141.0 -140.9 -140.8 ... -52.21 -52.12 -52.04
  * time     (time) float64 184kB 0.0 1.0 2.0 ... 2.301e+04 2.301e+04 2.301e+04
Data variables:
    pr       (time, lat, lon) float32 50GB ...
Attributes: (12/29)
    model_id:               PCIC_BLEND_v1
    domain:                 Canada
    CDO:                    Climate Data Operators version 1.9.7.1 (http://mp...
    _nc3_strict:            1
    NRCANmet_dataset:       ANUSPLIN Adjusted Daily Prec. interpolated Canada...
    NRCANmet_institution:   Canadian Forest Service, Natural Resources Canada
    ...                     ...
    contact:                Pacific Climate Impacts Consortium
    NRCANmet_references:    MacDonald H, McKenney DW, Wang XL, Pedlar J, Papa...
    NRCANmet_institute_id:  CFS-NRCan
    project_id:             other
    PNWNAmet_institute_id:  PCIC
    institute_id:           PCIC

In [4]:
print(ds)
list(ds.data_vars)
list(ds.coords)

<xarray.Dataset> Size: 50GB
Dimensions:  (lat: 510, lon: 1068, time: 23011)
Coordinates:
  * lat      (lat) float64 4kB 41.04 41.12 41.21 41.29 ... 83.29 83.38 83.46
  * lon      (lon) float64 9kB -141.0 -140.9 -140.8 ... -52.21 -52.12 -52.04
  * time     (time) float64 184kB 0.0 1.0 2.0 ... 2.301e+04 2.301e+04 2.301e+04
Data variables:
    pr       (time, lat, lon) float32 50GB ...
Attributes: (12/29)
    model_id:               PCIC_BLEND_v1
    domain:                 Canada
    CDO:                    Climate Data Operators version 1.9.7.1 (http://mp...
    _nc3_strict:            1
    NRCANmet_dataset:       ANUSPLIN Adjusted Daily Prec. interpolated Canada...
    NRCANmet_institution:   Canadian Forest Service, Natural Resources Canada
    ...                     ...
    contact:                Pacific Climate Impacts Consortium
    NRCANmet_references:    MacDonald H, McKenney DW, Wang XL, Pedlar J, Papa...
    NRCANmet_institute_id:  CFS-NRCan
    project_id:             other

['lat', 'lon', 'time']

### Load the CSV file from Yellowknife Airport

In [5]:
csv_path = project_root / paths["raw_data"]["station_precip"]

with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(15):
        print(f"{i+1}: {f.readline()}")


1: # CSV-File created with merge-csv.com

2: # -----------------------------------------------------------

3: 

4: x,y,STATION_NAME,CLIMATE_IDENTIFIER,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,MEAN_TEMPERATURE,MEAN_TEMPERATURE_FLAG,MIN_TEMPERATURE,MIN_TEMPERATURE_FLAG,MAX_TEMPERATURE,MAX_TEMPERATURE_FLAG,TOTAL_PRECIPITATION,TOTAL_PRECIPITATION_FLAG,TOTAL_RAIN,TOTAL_RAIN_FLAG,TOTAL_SNOW,TOTAL_SNOW_FLAG,SNOW_ON_GROUND,SNOW_ON_GROUND_FLAG,DIRECTION_MAX_GUST,DIRECTION_MAX_GUST_FLAG,SPEED_MAX_GUST,SPEED_MAX_GUST_FLAG,COOLING_DEGREE_DAYS,COOLING_DEGREE_DAYS_FLAG,HEATING_DEGREE_DAYS,HEATING_DEGREE_DAYS_FLAG,MIN_REL_HUMIDITY,MIN_REL_HUMIDITY_FLAG,MAX_REL_HUMIDITY,MAX_REL_HUMIDITY_FLAG,STN_ID

5: -114.44027777777778,62.46277777777778,YELLOWKNIFE A,2204100,2204100.1942.7.1,1942-07-01 00:00:00,NT,1942,7,1,18.1,,12.2,,23.9,,0,,0,,0,,,,,,,,0.1,,0,,,,,,1706

6: -114.44027777777778,62.46277777777778,YELLOWKNIFE A,2204100,2204100.1942.7.2,1942-07-02 00:00:00,NT,1942,7,2,15.6,,13.3,

In [9]:
df_station = pd.read_csv(
    resolve_relative_path(paths["raw_data"]["station_precip"]),
    comment="#",
)

df_station.columns
df_station[["LOCAL_DATE", "TOTAL_PRECIPITATION", "TOTAL_PRECIPITATION_FLAG"]].head()

,LOCAL_DATE,TOTAL_PRECIPITATION,TOTAL_PRECIPITATION_FLAG
0,1942-07-01 00:00:00,0.0,NaN
1,1942-07-02 00:00:00,1.5,NaN
2,1942-07-03 00:00:00,0.5,NaN
3,1942-07-04 00:00:00,0.0,NaN
4,1942-07-05 00:00:00,0.0,NaN


In [10]:
df_station["TOTAL_PRECIPITATION_FLAG"].value_counts(dropna=False)

TOTAL_PRECIPITATION_FLAG
NaN    19065
T       6552
E         90
M          5
Name: count, dtype: int64

For station precipitation data, trace precipitation (flag ‘T’) is treated as 0.0 mm; estimated values are retained; missing values are excluded from aggregation. 